### A5.4.9. Build Common Subexpression Elimination Pass

**Problem:**

Build a graph transformation pass that finds duplicate computations — nodes with the same operation and the same inputs — and replaces duplicates with a single shared node.

**Explanation:**

Common subexpression elimination (CSE) identifies nodes that compute the same thing and merges them. Two nodes are equivalent if they have the same operation type and the same input nodes (by identity). Keeping only one copy avoids redundant computation.

How to build it:

1. Walk the graph in topological order.
2. For each node, build a key: `(op_type, tuple of input node identities)`.
3. Keep a dictionary mapping keys to the first node seen with that key.
4. If the key already exists, the current node is a duplicate. Replace all references to the duplicate with the original node.
5. Remove the duplicate node from the graph.

**Example:**

If the graph computes `a + b` in two separate nodes, CSE replaces one with the other so the addition runs only once.

In [ ]:
class Node:
    def __init__(self, name, op_type, inputs=None):
        self.name = name
        self.op_type = op_type
        self.inputs = inputs or []

    def __repr__(self):
        input_names = [node.name for node in self.inputs]
        return f"{self.name}({self.op_type}, inputs={input_names})"


def eliminate_common_subexpressions(nodes):
    seen = {}
    replacements = {}

    for node in list(nodes):
        if node.op_type == "input":
            continue

        resolved_inputs = tuple(
            replacements.get(id(inp), inp)
            for inp in node.inputs
        )
        key = (node.op_type, tuple(id(inp) for inp in resolved_inputs))

        if key in seen:
            replacements[id(node)] = seen[key]
            nodes.remove(node)
        else:
            node.inputs = list(resolved_inputs)
            seen[key] = node

    for node in nodes:
        node.inputs = [
            replacements.get(id(inp), inp)
            for inp in node.inputs
        ]

    return nodes


a = Node("a", "input")
b = Node("b", "input")
add1 = Node("add1", "add", [a, b])
add2 = Node("add2", "add", [a, b])
mul1 = Node("mul1", "mul", [add1, b])
mul2 = Node("mul2", "mul", [add2, a])
out = Node("out", "add", [mul1, mul2])

all_nodes = [a, b, add1, add2, mul1, mul2, out]

print("Before CSE:")
for node in all_nodes:
    print(f"  {node}")

eliminate_common_subexpressions(all_nodes)

print("\nAfter CSE:")
for node in all_nodes:
    print(f"  {node}")

**References:**

📘 Aho, A. V. et al. (2006). *Compilers: Principles, Techniques, and Tools* — Common Subexpression Elimination

---

[⬅️ Previous: Build Constant Folding Pass](./08_build_constant_folding_pass.ipynb) | [Next: Build Layout Propagation Pass ➡️](./10_build_layout_propagation_pass.ipynb)